# 00 · Orquestrador do pipeline ETL

**MVP: Atrasos de entrega no e-commerce brasileiro (Olist)**
Universidade de Brasília · Departamento de Engenharia de Produção · Aluno: Filippe Dumont Lana (202024526)

Este notebook executa o pipeline **ponta a ponta**, em sequência e numa única sessão do Google Colab,
e registra cada etapa em um log persistido na nuvem.

```
Kaggle (9 CSVs) ──► 01 Coleta ──► BRONZE ──► 02 Transformação ──► SILVER ──► 03 Carga ──► GOLD (estrela)
                                                                                        │
                                                   05 Análise (P1–P6) ◄── 04 Qualidade ◄┘
```

| Camada | Conteúdo | Persistência |
|---|---|---|
| `raw/` | CSVs originais e manifesto da coleta (data, volume, checksums) | Google Drive |
| `bronze/` | Cópia fiel da origem, tudo como texto, mais colunas de linhagem | Google Drive · Delta Lake |
| `silver/` | Dados tipados, padronizados e conciliados (regras R01–R08) | Google Drive · Delta Lake |
| `gold/` | Esquema estrela, resultados de qualidade e indicadores (regras G01–G09) | Google Drive · Delta Lake |
| `logs/` | Log de execução de cada etapa | Google Drive · Delta Lake |

**Como executar:** *Ambiente de execução → Executar tudo*. Na primeira vez, autorize o acesso ao
Google Drive. O tempo total fica entre 10 e 15 minutos.

### Configuração do ambiente
Esta célula prepara o ambiente de execução. No Google Colab, ela:
1. clona o repositório do projeto, que contém o módulo `scripts/common.py`;
2. instala o PySpark e o Delta Lake;
3. monta o Google Drive, onde o lakehouse fica persistido.

Na primeira execução, o Colab pede autorização de acesso ao Drive.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/FilippeDumont/mvp-olist-atrasos-entrega.git"
REPO_DIR = "/content/repo"

if "google.colab" in sys.modules:
    if not os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub"], check=True)
    sys.path.insert(0, f"{REPO_DIR}/scripts")
else:  # execução local de teste
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))

import common as C

C.mount_drive()
spark = C.get_spark()
print("Spark", spark.version, "| Delta Lake habilitado")

In [ ]:
import time
NOTEBOOKS = ["01_coleta", "02_transformacao", "03_carga_gold", "04_qualidade", "05_analise"]
pasta_nb = os.path.join(C.REPO_ROOT, "notebooks")
t0 = time.time()
for nb in NOTEBOOKS:
    print(f"\n\n{'#' * 78}\n# Executando {nb}\n{'#' * 78}")
    get_ipython().run_line_magic("run", f"{pasta_nb}/{nb}.ipynb")
print(f"\nPipeline concluído em {(time.time() - t0) / 60:.1f} min | run_id {C.RUN_ID}")

## Log de execução desta rodada

In [ ]:
log = C.read_table(*C.LOG_TABLE)
log.filter(log.run_id == C.RUN_ID).orderBy("inicio_utc").select(
    "etapa", "destino", "linhas", "duracao_s", "status").show(60, truncate=False)

## Evidência de persistência na nuvem
A listagem abaixo mostra a estrutura do lakehouse **no Google Drive**, com o tamanho de cada tabela.
Em seguida aparece o histórico de versões Delta da tabela fato, em que cada carga gera uma nova versão.

In [ ]:
def tamanho(p):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs)

print(C.CLOUD_BASE)
for camada in ["raw", "bronze", "silver", "gold", "logs", "evidencias"]:
    base = f"{C.CLOUD_BASE}/{camada}"
    if not os.path.isdir(base):
        continue
    print(f"├── {camada}/  ({tamanho(base) / 1e6:,.1f} MB)")
    for t in sorted(os.listdir(base)):
        p = f"{base}/{t}"
        print(f"│   ├── {t}" + (f"  ({tamanho(p) / 1e6:,.2f} MB)" if os.path.isdir(p) else ""))

In [ ]:
C.table_history("gold", "fato_pedido").select("version", "timestamp", "operation", "operationMetrics.numOutputRows").show(truncate=False)